# Kiem chung chat luong model ONNX da convert

Notebook nay **tach rieng khoi `convert_to_onnx.ipynb`** — chi lam nhiem vu
kiem tra chat luong 1 file `.onnx` DA export san (khong export lai tu dau).
Dung sau khi da chay xong `convert_to_onnx.ipynb` va co file `.onnx` tren Drive.

Cac phep test trong nay:
1. Test tren anh that (upload/Drive) bang cach chia tile
2. Doi chieu voi anh demo chinh thuc `demo/blurry.jpg` cua repo
3. Phep thu quyet dinh: so sanh voi `NAFNetLocal` (kien truc TLC goc, PyTorch
   thuan, khong tile) — de biet ONNX+tile co lam giam chat luong so voi ban goc khong
4. Zoom vao 1 vung cu the de danh gia chinh xac (tranh bi "ao giac" do anh full-size
   bi thu nho khi hien thi)
5. Test bang dung anh trong tap test GoPro/SIDD chinh thuc (lmdb) — cach cong bang
   nhat de doi chieu voi benchmark PSNR/SSIM cua paper

> Luu y: file nay **khong dinh dang de tich hop app** — chi de ban tu kiem chung
> trong qua trinh phat trien/viet bao cao. App thuc te chi can dung dung file
> `.onnx` + buoc tien xu ly tile giong muc 1 o day, khong can toi lmdb hay NAFNetLocal.

## Buoc 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Buoc 2. Lay source code NAFNet + cai dependencies

Can `basicsr` de dung kien truc `NAFNetLocal` (dung cho phep thu quyet dinh o Buoc 5).

In [ ]:
%cd /content
!rm -rf NAFNet
!git clone https://github.com/megvii-research/NAFNet.git
%cd /content/NAFNet

!pip install -q -r requirements.txt
!pip install -q onnxruntime lmdb

In [ ]:
import sys
sys.path.insert(0, '/content/NAFNet')

import os
import torch
import numpy as np
import onnxruntime as ort
from PIL import Image
import matplotlib.pyplot as plt
from basicsr.models.archs.NAFNet_arch import NAFNetLocal

print('Torch:', torch.__version__)

## Buoc 3. CONFIG — nhap dung gia tri da dung khi convert

Copy chinh xac tu `convert_to_onnx.ipynb` (Buoc 3 cua file do): `CHECKPOINT_PATH`,
`PRESET`, `IMG_H`, `IMG_W`, va duong dan file `.onnx` da simplify.

In [ ]:
CHECKPOINT_PATH = '/content/drive/MyDrive/Thesis/Product/Data/NAFNet-GoPro-width64.pth'  # <-- khop voi convert_to_onnx.ipynb
PRESET = 'gopro_width64'  # <-- khop voi convert_to_onnx.ipynb
IMG_H, IMG_W = 256, 256   # <-- khop voi convert_to_onnx.ipynb

PRESETS = {
    'sidd_width32':  dict(width=32, enc_blk_nums=[2, 2, 4, 8],  middle_blk_num=12, dec_blk_nums=[2, 2, 2, 2]),
    'sidd_width64':  dict(width=64, enc_blk_nums=[2, 2, 4, 8],  middle_blk_num=12, dec_blk_nums=[2, 2, 2, 2]),
    'gopro_width32': dict(width=32, enc_blk_nums=[1, 1, 1, 28], middle_blk_num=1,  dec_blk_nums=[1, 1, 1, 1]),
    'gopro_width64': dict(width=64, enc_blk_nums=[1, 1, 1, 28], middle_blk_num=1,  dec_blk_nums=[1, 1, 1, 1]),
    'reds_width64':  dict(width=64, enc_blk_nums=[1, 1, 1, 28], middle_blk_num=1,  dec_blk_nums=[1, 1, 1, 1]),
}
MODEL_CFG = PRESETS[PRESET]

ONNX_SIMPLIFIED_PATH = f'/content/drive/MyDrive/NAFNet/nafnet_{PRESET}_{IMG_H}x{IMG_W}_sim.onnx'  # <-- sua neu khac

assert os.path.exists(ONNX_SIMPLIFIED_PATH), f'Khong tim thay file ONNX: {ONNX_SIMPLIFIED_PATH}. Kiem tra lai duong dan.'
assert os.path.exists(CHECKPOINT_PATH), f'Khong tim thay checkpoint: {CHECKPOINT_PATH}.'
print('OK, tim thay ca 2 file. Preset:', PRESET, MODEL_CFG)

## Buoc 4. Load ONNX session + build NAFNetLocal (de doi chieu)

In [ ]:
sess = ort.InferenceSession(ONNX_SIMPLIFIED_PATH, providers=['CPUExecutionProvider'])
print('Da load ONNX session:', ONNX_SIMPLIFIED_PATH)

ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu')
state_dict = ckpt.get('params', ckpt) if isinstance(ckpt, dict) else ckpt

model_local = NAFNetLocal(img_channel=3, **MODEL_CFG, train_size=(1, 3, 256, 256), fast_imp=False)
model_local.load_state_dict(state_dict, strict=True)
model_local.eval()
print('Da build NAFNetLocal (PyTorch, co TLC) va load checkpoint.')

## Buoc 5. Ham dung chung

`run_tiled_inference`: mo phong dung cach app mobile se chay (ONNX, chia tile
co dinh `IMG_H x IMG_W`, khong resize/khong meo ty le).

`run_local_full`: chay `NAFNetLocal` PyTorch tren TOAN BO anh, khong cat tile —
dung lam "ket qua tham chieu" (gan voi cach paper tao GIF demo nhat).

In [ ]:
def run_tiled_inference(image_path, sess, img_h, img_w):
    img = Image.open(image_path).convert('RGB')
    w, h = img.size
    img_np = np.array(img).astype(np.float32) / 255.0

    pad_h, pad_w = (-h) % img_h, (-w) % img_w
    img_padded = np.pad(img_np, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect')
    H_pad, W_pad = img_padded.shape[:2]
    out_padded = np.zeros_like(img_padded)

    n_tiles_h, n_tiles_w = H_pad // img_h, W_pad // img_w
    print(f'Anh {w}x{h} -> pad ve {W_pad}x{H_pad} -> {n_tiles_w}x{n_tiles_h} = {n_tiles_w*n_tiles_h} tile')

    for i in range(n_tiles_h):
        for j in range(n_tiles_w):
            y0, y1 = i * img_h, (i + 1) * img_h
            x0, x1 = j * img_w, (j + 1) * img_w
            tile_in = img_padded[y0:y1, x0:x1, :].transpose(2, 0, 1)[None].astype(np.float32)
            tile_out = sess.run(None, {'input': tile_in})[0]
            out_padded[y0:y1, x0:x1, :] = tile_out[0].transpose(1, 2, 0)

    out = np.clip(out_padded[:h, :w, :], 0, 1)
    return img_np, out


def run_local_full(image_np, model_local):
    tensor = torch.from_numpy(image_np.transpose(2, 0, 1)).unsqueeze(0).float()
    with torch.no_grad():
        out = model_local(tensor)
    return out[0].permute(1, 2, 0).clamp(0, 1).numpy()


def show_and_report(img_in, img_out, title_in, title_out):
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    axes[0].imshow(img_in); axes[0].set_title(title_in); axes[0].axis('off')
    axes[1].imshow(img_out); axes[1].set_title(title_out); axes[1].axis('off')
    plt.show()
    mean_diff = np.abs(img_in - img_out).mean()
    print(f'Mean abs diff input/output: {mean_diff:.5f} (thang [0,1])')
    return mean_diff


print('Da dinh nghia ham dung chung.')

## Buoc 6. Test tren anh that (chia tile, khong resize)

Chon 1 trong 2 cach dua anh vao Colab:

In [ ]:
# ==== Cach A: upload anh tu may tinh ====
from google.colab import files

uploaded = files.upload()
INPUT_IMAGE_PATH = list(uploaded.keys())[0]
print('Da nhan anh:', INPUT_IMAGE_PATH)

In [ ]:
# ==== Cach B: dung anh co san tren Drive (bo qua neu da dung Cach A) ====
DRIVE_IMAGE_PATH = '/content/drive/MyDrive/NAFNet/test_images/anh_test.jpg'  # <-- sua duong dan that

INPUT_IMAGE_PATH = DRIVE_IMAGE_PATH
print('Dung anh:', INPUT_IMAGE_PATH)

In [ ]:
img_in, img_out = run_tiled_inference(INPUT_IMAGE_PATH, sess, IMG_H, IMG_W)
mean_diff_user_img = show_and_report(img_in, img_out, 'Input goc', 'Output (ONNX + tile)')

OUTPUT_IMAGE_PATH = '/content/drive/MyDrive/NAFNet/test_images/ket_qua_onnx_full.png'
os.makedirs(os.path.dirname(OUTPUT_IMAGE_PATH), exist_ok=True)
Image.fromarray((img_out * 255).astype(np.uint8)).save(OUTPUT_IMAGE_PATH)
print('Da luu ket qua:', OUTPUT_IMAGE_PATH)

if mean_diff_user_img < 0.005:
    print('=> RAT GAN 0: model gan nhu khong thay doi gi anh nay ca.')
else:
    print('=> Model co thay doi anh. Xem tiep Buoc 7-8 de doi chieu voi ket qua chinh thuc,')
    print('   va Buoc 9 de zoom kiem tra chi tiet (mat thuong de bo sot cai thien nho).')

## Buoc 7. Doi chieu voi anh demo chinh thuc `demo/blurry.jpg`

Tach biet 2 kha nang: (a) anh cua ban nam ngoai phan phoi du lieu train
(domain gap), hay (b) chinh pipeline ONNX+tile dang co van de.

In [ ]:
import urllib.request

DEMO_URL = 'https://raw.githubusercontent.com/megvii-research/NAFNet/main/demo/blurry.jpg'
DEMO_LOCAL_PATH = '/content/demo_blurry.jpg'
urllib.request.urlretrieve(DEMO_URL, DEMO_LOCAL_PATH)
print('Da tai:', DEMO_LOCAL_PATH)

img_demo_in, img_demo_out_tile = run_tiled_inference(DEMO_LOCAL_PATH, sess, IMG_H, IMG_W)
mean_diff_demo_tile = show_and_report(img_demo_in, img_demo_out_tile, 'demo/blurry.jpg goc', 'Output (ONNX + tile)')

## Buoc 8. Phep thu quyet dinh: NAFNetLocal (PyTorch thuan, KHONG tile, KHONG ONNX)

- Neu cach nay cho ket qua net ro ret hon han Buoc 7 -> viec bo TLC + cat tile
  (danh doi de chay duoc tren mobile) la nguyen nhan giam chat luong.
- Neu CUNG cho ket qua yeu tuong tu -> nghi van chuyen sang checkpoint `.pth`
  (kiem tra lai da tai dung file/dung link chinh thuc chua).

In [ ]:
out_local_np = run_local_full(img_demo_in, model_local)
mean_diff_local = show_and_report(
    img_demo_in, out_local_np,
    'demo/blurry.jpg goc', 'Output NAFNetLocal (PyTorch, full anh, khong tile)'
)

print(f'\nSo sanh truc tiep tren cung anh demo:')
print(f'  ONNX + tile      : mean abs diff = {mean_diff_demo_tile:.5f}')
print(f'  NAFNetLocal full  : mean abs diff = {mean_diff_local:.5f}')
if mean_diff_local > mean_diff_demo_tile * 1.5:
    print('=> NAFNetLocal cai thien ro ret hon -> bo TLC + cat tile la nguyen nhan giam chat luong.')
else:
    print('=> Hai cach tuong tu nhau -> kiem tra lai nguon checkpoint .pth, hoac xem tiep Buoc 9')
    print('   (zoom) truoc khi ket luan — co the ca hai deu dang cai thien that, chi kho thay bang mat.')

## Buoc 9. Zoom vao 1 vung cu the de danh gia dung

`mean_abs_diff` tren toan anh lon (vi du 1280x720) de bi hieu nham la "gan 0"
khi thuc ra van co cai thien that, chi la kho thay khi anh bi thu nho de hien thi
canh nhau. Crop dung vung bi mo manh nhat o kich thuoc pixel day du de kiem tra lai.

In [ ]:
# Chinh CROP_Y, CROP_X, CROP_SIZE cho trung vung bi mo manh nhat trong anh dang test
# (vi du: nguoi dang di, mep ao, banh xe...). Toa do (0,0) la goc tren-trai.
CROP_Y, CROP_X, CROP_SIZE = 250, 200, 300  # <-- chinh lai cho khop anh dang test

crop_in = img_demo_in[CROP_Y:CROP_Y+CROP_SIZE, CROP_X:CROP_X+CROP_SIZE, :]
crop_out = out_local_np[CROP_Y:CROP_Y+CROP_SIZE, CROP_X:CROP_X+CROP_SIZE, :]

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(crop_in); axes[0].set_title('Input - ZOOM (pixel that)'); axes[0].axis('off')
axes[1].imshow(crop_out); axes[1].set_title('Output NAFNetLocal - ZOOM'); axes[1].axis('off')
plt.show()

print('Neu zoom van khong thay khac biet ro ret o vien/canh -> model that su khong cai thien duoc anh nay.')
print('Neu zoom thay net hon ro ret -> van de chi do hien thi anh full-size bi thu nho, khong phai loi ky thuat.')

## Buoc 10. Test bang dung anh trong tap test chinh thuc (lmdb)

Cach test cong bang nhat: dung dung anh + dung do phan giai ma paper dung de
bao cao PSNR/SSIM, loai bo hoan toan nghi van domain-gap.

Truoc khi chay: tai `input.lmdb` tu link chinh thuc trong `docs/GoPro.md`
(hoac `docs/SIDD.md` neu dang test model denoise), upload len Drive, sua
`LMDB_DIR` ben duoi cho dung duong dan.

Link test set GoPro chinh thuc: https://drive.google.com/file/d/1abXSfeRGrzj2mQ2n2vIBHtObU6vXvr7C/view

In [ ]:
import lmdb
import cv2

LMDB_DIR = '/content/drive/MyDrive/NAFNet/GoPro_test/input.lmdb'  # <-- sua duong dan that
SAMPLE_INDEX = 0  # doi so nay de thu anh khac trong tap test

meta_info_path = os.path.join(LMDB_DIR, 'meta_info.txt')
with open(meta_info_path, 'r') as f:
    keys = [line.split('.png')[0] + '.png' for line in f.readlines()]
print(f'Tap test co {len(keys)} anh. Dang lay anh: {keys[SAMPLE_INDEX]}')

env = lmdb.open(LMDB_DIR, readonly=True, lock=False, readahead=False)
with env.begin(write=False) as txn:
    value_buf = txn.get(keys[SAMPLE_INDEX].encode('ascii'))
env.close()

img_array = np.frombuffer(value_buf, dtype=np.uint8)
img_bgr = cv2.imdecode(img_array, cv2.IMREAD_UNCHANGED)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

GOPRO_TEST_IMAGE_PATH = '/content/gopro_test_sample.png'
Image.fromarray(img_rgb).save(GOPRO_TEST_IMAGE_PATH)
print('Da luu anh test chinh thuc:', GOPRO_TEST_IMAGE_PATH, '| kich thuoc:', img_rgb.shape)

plt.figure(figsize=(8, 5))
plt.imshow(img_rgb)
plt.title(f'Anh test chinh thuc: {keys[SAMPLE_INDEX]}')
plt.axis('off')
plt.show()

In [ ]:
# ==== Trich anh GOC SACH tuong ung tu target.lmdb (dung de tinh PSNR/SSIM that) ====
TARGET_LMDB_DIR = '/content/drive/MyDrive/NAFNet/GoPro_test/target.lmdb'  # <-- sua duong dan that

env_gt = lmdb.open(TARGET_LMDB_DIR, readonly=True, lock=False, readahead=False)
with env_gt.begin(write=False) as txn:
    gt_value_buf = txn.get(keys[SAMPLE_INDEX].encode('ascii'))  # dung CHUNG key voi input o cell tren
env_gt.close()

gt_array = np.frombuffer(gt_value_buf, dtype=np.uint8)
gt_bgr = cv2.imdecode(gt_array, cv2.IMREAD_UNCHANGED)
gt_rgb = cv2.cvtColor(gt_bgr, cv2.COLOR_BGR2RGB)
img_gopro_gt = gt_rgb.astype(np.float32) / 255.0

print('Da lay anh ground-truth tuong ung, kich thuoc:', gt_rgb.shape)
plt.figure(figsize=(8, 5))
plt.imshow(gt_rgb)
plt.title(f'Ground truth (target.lmdb): {keys[SAMPLE_INDEX]}')
plt.axis('off')
plt.show()

In [ ]:
from basicsr.metrics.psnr_ssim import calculate_psnr, calculate_ssim

img_gopro_in, img_gopro_out_tile = run_tiled_inference(GOPRO_TEST_IMAGE_PATH, sess, IMG_H, IMG_W)
mean_diff_gopro_tile = show_and_report(img_gopro_in, img_gopro_out_tile, 'Anh test chinh thuc', 'Output (ONNX + tile)')

out_gopro_local_np = run_local_full(img_gopro_in, model_local)
mean_diff_gopro_local = show_and_report(img_gopro_in, out_gopro_local_np, 'Anh test chinh thuc', 'Output NAFNetLocal (full anh)')

# ==== PSNR / SSIM that, so voi ground truth (target.lmdb) ====
# crop_border=0, test_y_channel=False -> giong dung config trong options/test/GoPro/NAFNet-width64.yml
psnr_input = calculate_psnr(img_gopro_in, img_gopro_gt, crop_border=0, test_y_channel=False)
psnr_tile = calculate_psnr(img_gopro_out_tile, img_gopro_gt, crop_border=0, test_y_channel=False)
psnr_local = calculate_psnr(out_gopro_local_np, img_gopro_gt, crop_border=0, test_y_channel=False)

ssim_input = calculate_ssim(img_gopro_in, img_gopro_gt, crop_border=0, test_y_channel=False)
ssim_tile = calculate_ssim(img_gopro_out_tile, img_gopro_gt, crop_border=0, test_y_channel=False)
ssim_local = calculate_ssim(out_gopro_local_np, img_gopro_gt, crop_border=0, test_y_channel=False)

print(f'\nTOM TAT mean_abs_diff (input vs output, khong can ground truth):')
print(f'  ONNX + tile      : {mean_diff_gopro_tile:.5f}')
print(f'  NAFNetLocal full  : {mean_diff_gopro_local:.5f}')

print(f'\nPSNR / SSIM THAT (so voi ground truth trong target.lmdb):')
print(f'  Input mo (chua xu ly)   : PSNR = {psnr_input:.2f} dB | SSIM = {ssim_input:.4f}')
print(f'  Output ONNX + tile      : PSNR = {psnr_tile:.2f} dB | SSIM = {ssim_tile:.4f}')
print(f'  Output NAFNetLocal full : PSNR = {psnr_local:.2f} dB | SSIM = {ssim_local:.4f}')
print(f'\n(De so sanh: paper bao cao NAFNet-GoPro-width64 dat trung binh ~33.71 dB tren toan bo tap test,')
print(f' con so tren chi la 1 anh mau nen co the lech it nhieu so voi trung binh benchmark.)')

In [ ]:
import time
from basicsr.metrics.psnr_ssim import calculate_psnr, calculate_ssim

N_SAMPLES = 20          # so anh dau tien can danh gia; dat = None de chay het ca tap test
EVAL_NAFNETLOCAL = False  # True neu muon danh gia them ban NAFNetLocal (cham hon nhieu tren CPU)


def decode_lmdb_image(env, key):
    with env.begin(write=False) as txn:
        buf = txn.get(key.encode('ascii'))
    bgr = cv2.imdecode(np.frombuffer(buf, dtype=np.uint8), cv2.IMREAD_UNCHANGED)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return rgb.astype(np.float32) / 255.0


def tile_infer_array(img_np, sess, img_h, img_w):
    """Giong run_tiled_inference nhung nhan thang mang numpy (khong doc lai tu file)."""
    h, w = img_np.shape[:2]
    pad_h, pad_w = (-h) % img_h, (-w) % img_w
    img_padded = np.pad(img_np, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect')
    H_pad, W_pad = img_padded.shape[:2]
    out_padded = np.zeros_like(img_padded)
    for i in range(H_pad // img_h):
        for j in range(W_pad // img_w):
            y0, y1 = i * img_h, (i + 1) * img_h
            x0, x1 = j * img_w, (j + 1) * img_w
            tile_in = img_padded[y0:y1, x0:x1, :].transpose(2, 0, 1)[None].astype(np.float32)
            tile_out = sess.run(None, {'input': tile_in})[0]
            out_padded[y0:y1, x0:x1, :] = tile_out[0].transpose(1, 2, 0)
    return np.clip(out_padded[:h, :w, :], 0, 1)


sample_keys = keys if N_SAMPLES is None else keys[:N_SAMPLES]
print(f'Danh gia tren {len(sample_keys)} / {len(keys)} anh cua tap test...')

env_in = lmdb.open(LMDB_DIR, readonly=True, lock=False, readahead=False)
env_gt = lmdb.open(TARGET_LMDB_DIR, readonly=True, lock=False, readahead=False)

psnr_input_list, ssim_input_list = [], []
psnr_tile_list, ssim_tile_list = [], []
psnr_local_list, ssim_local_list = [], []

t0 = time.time()
for idx, key in enumerate(sample_keys):
    img_in_np = decode_lmdb_image(env_in, key)
    img_gt_np = decode_lmdb_image(env_gt, key)

    out_tile_np = tile_infer_array(img_in_np, sess, IMG_H, IMG_W)

    psnr_in = calculate_psnr(img_in_np, img_gt_np, crop_border=0)
    psnr_t = calculate_psnr(out_tile_np, img_gt_np, crop_border=0)
    ssim_in = calculate_ssim(img_in_np, img_gt_np, crop_border=0)
    ssim_t = calculate_ssim(out_tile_np, img_gt_np, crop_border=0)

    psnr_input_list.append(psnr_in); ssim_input_list.append(ssim_in)
    psnr_tile_list.append(psnr_t); ssim_tile_list.append(ssim_t)

    log_line = f'[{idx+1}/{len(sample_keys)}] {key}: PSNR input={psnr_in:.2f} tile={psnr_t:.2f}'

    if EVAL_NAFNETLOCAL:
        out_local_np = run_local_full(img_in_np, model_local)
        psnr_l = calculate_psnr(out_local_np, img_gt_np, crop_border=0)
        ssim_l = calculate_ssim(out_local_np, img_gt_np, crop_border=0)
        psnr_local_list.append(psnr_l); ssim_local_list.append(ssim_l)
        log_line += f' local={psnr_l:.2f}'

    print(log_line)

env_in.close()
env_gt.close()

elapsed = time.time() - t0
print(f'\nThoi gian chay: {elapsed:.1f}s cho {len(sample_keys)} anh ({elapsed/len(sample_keys):.1f}s/anh)')

print(f'\n=== KET QUA TRUNG BINH tren {len(sample_keys)} anh ===')
print(f'Input (chua xu ly)   : PSNR = {np.mean(psnr_input_list):.2f} dB | SSIM = {np.mean(ssim_input_list):.4f}')
print(f'ONNX + tile          : PSNR = {np.mean(psnr_tile_list):.2f} dB | SSIM = {np.mean(ssim_tile_list):.4f}')
if EVAL_NAFNETLOCAL:
    print(f'NAFNetLocal full     : PSNR = {np.mean(psnr_local_list):.2f} dB | SSIM = {np.mean(ssim_local_list):.4f}')
print(f'\n(De so sanh: paper bao cao NAFNet-GoPro-width64 dat trung binh ~33.71 dB tren toan bo {len(keys)} anh test.)')

## Buoc 10b. Danh gia trung binh tren NHIEU anh (batch evaluation)

Buoc 10 chi test dung 1 anh (`SAMPLE_INDEX=0`) — PSNR tung anh dao dong nhieu,
khong dai dien cho ca tap test. Cell duoi day duyet qua N anh dau tien trong
`input.lmdb`/`target.lmdb`, tinh PSNR/SSIM trung binh — con so nay moi thuc su
so sanh duoc voi benchmark `33.71 dB` cua paper (paper tinh tren toan bo ~1111 anh).

Luu y: chay tren CPU, moi anh 1280x720 can ~15 tile (~15 lan forward). Voi
`N_SAMPLES=20` mat khoang vai phut; neu muon chay het ca tap test (`N_SAMPLES=None`)
co the mat hang gio tren CPU — nen thu voi so luong nho truoc.

## Ket luan

Doi chieu cac so `mean_abs_diff` va anh zoom o tren de danh gia:
- Neu ONNX+tile ~ NAFNetLocal tren moi anh -> convert dung, chat luong khong bi
  giam boi viec bo TLC/chia tile.
- Neu chenh lech lon -> can nang cap pipeline mobile (vi du: tile chong lap +
  blend bien, hoac xap xi TLC) truoc khi dua vao app.
- `mean_abs_diff` nho tren toan anh lon KHONG dong nghia model "khong lam gi" —
  luon zoom vao 1 vung cu the (Buoc 9) truoc khi ket luan.